# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore, process, and analyze the FAIR<sup>2</sup> protein abundance dataset (Croissant schema).

### Dataset Source
The dataset source is provided via a Croissant schema URL at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant dataset using `mlcroissant`. This also prints basic dataset summary information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")
# Show publication date, version, authors and license
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview

In this section, we'll review the available record sets, fields, and relevant `@id` values so you can refer to each entity unambiguously in later sections.

> **NOTE:** All field, record set, and column references will be via their Croissant `@id` attribute.

Let's enumerate all top-level record sets with their IDs and list their fields.

In [ ]:
# View all record sets: each is uniquely referenced by its `@id`

record_sets = list(dataset.record_sets)
print(f"There are {len(record_sets)} record sets in the dataset.\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', None)})")
    print()

## 3. Data Extraction

Now we'll load data from all available record sets using `mlcroissant`. Each DataFrame is keyed by its record set `@id`, and fields can be referred to by their `@id` for filtering and processing later.

In [ ]:
# Extract data from all record sets to DataFrames, mapping by their `@id`

dataframes = {}
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet '{rs.name}' (@id={rs.id})")
    if len(records) > 0:
        print(f"Columns: {dataframes[rs.id].columns.tolist()}")
    print()
# Let's inspect the first record set and preview rows
primary_record_set_id = record_sets[0].id if len(record_sets) > 0 else None
if primary_record_set_id:
    print(f"Preview of data for RecordSet '@id={primary_record_set_id}':")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process data from the primary record set. We will select a typical numeric field for analysis, filter records, normalize its values, and, if present, group by a key attribute.

All references below use `@id` attributes.

> **TIP:** Replace values in the variables below with those shown in the overview above _when you know the specific `@id`_.

In [ ]:
# --- EDA Variables ---
# Choose the main record set (use @id from earlier output)
record_set_id = primary_record_set_id  # update if needed
# Set the field @id for the main numeric field (use exact string from field list)
numeric_field_id = None
# Set group field for aggregation (use suitable @id or None)
group_field_id = None

# Recommend a numeric field by searching common field names
if record_set_id is not None:
    candidates = ['coverage', 'MW', 'abundance', 'peptide', 'score', 'count']
    for col in dataframes[record_set_id].columns:
        for cand in candidates:
            if cand.lower() in col.lower():
                numeric_field_id = col
                break
        if numeric_field_id:
            break
    # For grouping, favor 'description', 'category', 'sample', etc.
    group_candidates = ['description', 'sample', 'group', 'category', 'class']
    for col in dataframes[record_set_id].columns:
        for cand in group_candidates:
            if cand.lower() in col.lower():
                group_field_id = col
                break
        if group_field_id:
            break

print(f"Selected numeric field for EDA: {numeric_field_id}")
print(f"Selected group field for EDA: {group_field_id}")

df = dataframes[record_set_id]
# Convert to numeric for analysis
if numeric_field_id is not None:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median() if not pd.isna(df[numeric_field_id].median()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (showing group means):")
        display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and if available, show its breakdown by group. (Requires `matplotlib` and/or `seaborn`.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id is not None:
    # Basic distribution
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (@id)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group field exists, show boxplot or grouped bar plot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        # Show only top groups
        top_groups = df[group_field_id].value_counts().nlargest(8).index
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}' (@id)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

* You loaded and explored the FAIR<sup>2</sup> Mass Spectrometry dataset using `mlcroissant`.
* The notebook illustrated how to reference all data entities via their Croissant `@id`, ensuring reproducible and unambiguous data access.
* You performed preliminary data filtering and normalization on a numeric field and created both overall and grouped visualizations.

### Next Steps
* For deeper biological or proteomic analysis, investigate field and record set definitions in detail using their `@id`, and map those to your research or machine learning workflow.
* Use advanced filtering, normalization, or machine learning models directly on the extracted DataFrames.
* Share this notebook, recycling its approach for any Croissant schema-compliant dataset!